# Matricea de confuzie și curbele ROC

Material ONIA — Resurse educaționale. Exercițiile pot fi modificate și executate în JupyterLab.

## De la probabilitate la etichetă

Un clasificator binar poate atribui un scor de probabilitate p = 0.73 pentru clasa pozitivă. Folosind pragul 0,5 prezicem clasa 1, dar cu pragul 0,8 prezicem clasa 0. Datele și modelul nu s-au schimbat; s-a schimbat doar regula de decizie. Un prag mare tinde să reducă alertele false, dar poate rata pozitive; unul mic tinde să surprindă mai multe pozitive, dar crește adesea alertele false. Efectul concret depinde de distribuția scorurilor.

In [ ]:
scoruri = [0.15, 0.42, 0.73, 0.91]
for prag in [0.5, 0.8]:
    pred = [int(p >= prag) for p in scoruri]
    print(prag, pred)

## Construirea matricii pas cu pas

Pentru fiecare observație comparăm eticheta reală cu predicția. Dacă ambele sunt 1, creștem TP. Dacă real=0 și pred=1, creștem FP. Dacă real=1 și pred=0, creștem FN. În rest, TN. O matrice de confuzie nu are mereu aceeași orientare grafică în toate bibliotecile; citim întotdeauna etichetele axelor. Convenția folosită aici: rânduri = clase reale, coloane = clase prezise.

In [ ]:
reale = [1, 0, 1, 1, 0, 0]
prezise = [1, 0, 0, 1, 1, 0]
tp = fp = fn = tn = 0
for y, p in zip(reale, prezise):
    if y == 1 and p == 1:
        tp += 1
    elif y == 0 and p == 1:
        fp += 1
    elif y == 1 and p == 0:
        fn += 1
    else:
        tn += 1
print(tp, fp, fn, tn)

## Sensitivity, specificity și false positive rate

TPR = Recall = TP/(TP + FN) TNR = TN/(TN + FP) FPR = FP/(FP + TN) TPR (sensibilitatea) este fracția cazurilor pozitive descoperite. TNR (specificitatea) este fracția cazurilor negative recunoscute. FPR este fracția negativelor confundate cu pozitive și este egal cu 1 − TNR . Exemplu: în detectarea unui defect, să avem TP=30, FN=10, FP=5, TN=55. Recall=30/40=0,75; FPR=5/60≈0,083.

In [ ]:
tp, fn, fp, tn = 30, 10, 5, 55
print('Recall/TPR:', tp / (tp + fn))
print('Specificity:', tn / (tn + fp))
print('FPR:', fp / (fp + tn))

## Ce se întâmplă când deplasăm pragul?

Dacă pragul crește, numărul de predicții pozitive nu poate crește pentru aceleași scoruri. De regulă, TP și FP scad sau rămân constante, FN și TN cresc sau rămân constante. De aceea există un compromis între detectarea pozitivelor și alarmele false. Acest fenomen este esențial pentru probleme în care costul unei alarme false diferă de costul ratării unui pozitiv. În laborator, mută pragul și urmărește exact ce puncte își schimbă predicția.

## Curba ROC pas cu pas

ROC se construiește variind pragul peste valorile scorurilor modelului. Pe axa orizontală este FPR , iar pe axa verticală TPR . Un model care separă bine pozitivele de negative urcă rapid spre colțul stânga-sus. Linia diagonală reprezintă un comportament de referință asemănător unei ordonări aleatoare. Fiecare punct de pe ROC corespunde unui anumit prag; nu trebuie interpretată ca o funcție de timp sau de numărul de iterații.

In [ ]:
from sklearn.metrics import roc_curve
reale = [0, 1, 0, 1, 0, 1]
scoruri = [0.12, 0.89, 0.31, 0.72, 0.60, 0.58]
fpr, tpr, praguri = roc_curve(reale, scoruri)
for a, b, t in zip(fpr, tpr, praguri):
    print(round(a, 2), round(b, 2), round(float(t), 3))

## ROC-AUC: calitatea ordonării

AUC este aria de sub curba ROC. Pentru clasificare binară, poate fi interpretată și ca probabilitatea ca un exemplu pozitiv ales aleator să primească un scor mai mare decât unul negativ, ținând cont de egalități. 1 înseamnă separare perfectă după scor; 0,5 reprezintă un nivel de referință similar ordonării aleatoare. ROC-AUC nu stabilește automat pragul potrivit și nu măsoară calibrarea probabilităților. Un model cu AUC bun poate furniza probabilități prost calibrate.

In [ ]:
from sklearn.metrics import roc_auc_score
reale = [0, 1, 0, 1, 0, 1]
scoruri = [0.2, 0.9, 0.3, 0.7, 0.8, 0.6]
print(roc_auc_score(reale, scoruri))

## Curba Precision–Recall pentru clase rare

Când numărul cazurilor pozitive este foarte mic, ROC poate ascunde costul unui număr aparent mic de false alarme. În astfel de situații analizăm curba precision–recall, cu precision pe verticală și recall pe orizontală. average_precision_score este o sumarizare uzuală; nu este identică în toate implementările cu aria calculată prin metoda trapezelor. În competiții cu date dezechilibrate trebuie verificată exact definiția metricii de evaluare, inclusiv media macro/micro și eticheta pozitivă.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
reale = [0, 0, 0, 0, 1, 0, 1, 0]
scoruri = [.12, .32, .20, .73, .87, .26, .61, .15]
precision, recall, thresholds = precision_recall_curve(reale, scoruri)
print('Average precision:', average_precision_score(reale, scoruri))
print('Puncte:', list(zip(precision.round(2), recall.round(2))))

## F1 nu este constant când schimbăm pragul

Precision și recall se schimbă în general la fiecare salt peste scorul unui exemplu. Prin urmare, și F1 se modifică. Pragul 0,5 nu maximizează întotdeauna F1 și nu este optim pentru orice aplicație. Nu selectăm pragul pe setul de test final : îl alegem pe un set de validare sau prin cross-validation. În probleme cu costuri reale (de exemplu, alarmare), alegem pragul după costul erorilor sau după o restricție precum recall minim.

In [ ]:
from sklearn.metrics import f1_score
y_true = [0, 1, 0, 1, 0, 1]
p = [.25, .47, .52, .72, .28, .86]
for prag in [.3, .5, .7]:
    y_pred = [int(x >= prag) for x in p]
    print(prag, f1_score(y_true, y_pred))

## Matricea de confuzie multiclasă

Pentru trei clase A, B și C, matricea are trei rânduri și trei coloane. Elementul din poziția (i,j) arată câte observații din clasa reală i au fost prezise drept j. Elementele de pe diagonală reprezintă clasificări corecte. Pentru precision/recall pe o clasă putem trata clasa respectivă drept pozitivă și toate celelalte drept negative (One-vs-Rest). classification_report oferă rapoarte pentru fiecare clasă și medii macro/weighted.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
real = ['A', 'A', 'B', 'B', 'C', 'C']
pred = ['A', 'B', 'B', 'C', 'C', 'A']
print(confusion_matrix(real, pred, labels=['A', 'B', 'C']))
print(classification_report(real, pred, zero_division=0))

## Când evaluăm mai degrabă o probabilitate?

Acuratețea privește doar eticheta finală. Pentru sisteme în care incertitudinea este importantă, trebuie evaluate și probabilitățile: log-loss și Brier score sunt două exemple. Calibrarea răspunde la întrebarea: dintre observațiile cărora le atribui probabilitate 0,7, aproximativ 70% sunt într-adevăr pozitive? Este posibil ca un model să aibă accuracy mare, dar probabilități prost calibrate. Un scor ROC-AUC mare nu garantează calibrarea.

In [ ]:
from sklearn.metrics import brier_score_loss, log_loss
y = [0, 1, 1, 0]
p = [0.2, 0.9, 0.7, 0.6]
print('Brier:', brier_score_loss(y, p))
print('Log-loss:', log_loss(y, p))